# Colab Runner\n\nUse this notebook as the compute bridge for Codex. Codex edits the repo locally, you push/pull via GitHub, and this notebook runs the current code on a Colab runtime while saving logs and result metadata under `results/`.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
import os\nfrom pathlib import Path\n\nREPO_URL = 'https://github.com/Rajshree-Mittal/01.git'\nREPO_DIR = Path('/content/01')\n\nif not REPO_DIR.exists():\n    !git clone {REPO_URL} {REPO_DIR}\nelse:\n    %cd {REPO_DIR}\n    !git pull origin main\n\n%cd {REPO_DIR}\nprint('Repo ready at', REPO_DIR)

In [ ]:
!pip install -q -r requirements.txt

## Data Location\n\nPlace large datasets in Google Drive, then copy or symlink them into this runtime. The current scripts expect these paths when relevant:\n\n- `bot-iot/*.csv` for `data_balancing.py` and `comparision.py`\n- `bot-iot_final.csv` for `comparision.py`\n- `processed_datasets/ton-iot_final.csv` for `testing_data_Xgboost.py`

In [ ]:
from pathlib import Path\nimport os\n\nRESULTS_DIR = Path('results')\nRESULTS_DIR.mkdir(exist_ok=True)\nPath('plots').mkdir(exist_ok=True)\n\n# Optional: set this to a Drive folder that contains your datasets.\n# Example: DRIVE_DATA_DIR = Path('/content/drive/MyDrive/pida_data')\nDRIVE_DATA_DIR = None\n\nif DRIVE_DATA_DIR is not None and DRIVE_DATA_DIR.exists():\n    if (DRIVE_DATA_DIR / 'bot-iot').exists() and not Path('bot-iot').exists():\n        os.symlink(DRIVE_DATA_DIR / 'bot-iot', 'bot-iot')\n    if (DRIVE_DATA_DIR / 'processed_datasets').exists() and not Path('processed_datasets').exists():\n        os.symlink(DRIVE_DATA_DIR / 'processed_datasets', 'processed_datasets')\n    if (DRIVE_DATA_DIR / 'bot-iot_final.csv').exists() and not Path('bot-iot_final.csv').exists():\n        os.symlink(DRIVE_DATA_DIR / 'bot-iot_final.csv', 'bot-iot_final.csv')\n\nexpected_paths = {\n    'bot_iot_folder': Path('bot-iot'),\n    'bot_iot_final': Path('bot-iot_final.csv'),\n    'ton_iot_final': Path('processed_datasets/ton-iot_final.csv'),\n}\n\nfor name, path in expected_paths.items():\n    print(f'{name}: {path} -> {path.exists()}')

In [ ]:
import json\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\ndef run_step(name, command, required_paths=()):\n    missing = [str(Path(p)) for p in required_paths if not Path(p).exists()]\n    log_path = RESULTS_DIR / f'{name}.log'\n    meta_path = RESULTS_DIR / f'{name}.json'\n\n    if missing:\n        message = f'SKIPPED {name}: missing required paths: {missing}'\n        print(message)\n        log_path.write_text(message + '\\n')\n        meta = {'name': name, 'status': 'skipped', 'missing': missing, 'command': command}\n        meta_path.write_text(json.dumps(meta, indent=2))\n        return meta\n\n    start = time.time()\n    print(f'Running {name}: {command}')\n    proc = subprocess.run(command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)\n    duration = time.time() - start\n    log_path.write_text(proc.stdout)\n    print(proc.stdout[-4000:])\n\n    meta = {\n        'name': name,\n        'status': 'passed' if proc.returncode == 0 else 'failed',\n        'returncode': proc.returncode,\n        'duration_seconds': round(duration, 2),\n        'command': command,\n        'log_path': str(log_path),\n    }\n    meta_path.write_text(json.dumps(meta, indent=2))\n    if proc.returncode != 0:\n        raise RuntimeError(f'{name} failed. See {log_path}')\n    return meta

In [ ]:
# Step 1: generate bot-iot_final.csv from raw bot-iot CSV files.\n# This can be slow because it trains the contrastive encoder.\nRUN_DATA_BALANCING = False\n\nif RUN_DATA_BALANCING:\n    run_step('data_balancing', f'{sys.executable} data_balancing.py', required_paths=['bot-iot'])\nelse:\n    print('Skipped data_balancing.py. Set RUN_DATA_BALANCING = True when raw bot-iot data is available.')

In [ ]:
# Step 2: compare original and processed distributions.\nRUN_COMPARISON = False\n\nif RUN_COMPARISON:\n    run_step('comparison', f'{sys.executable} comparision.py', required_paths=['bot-iot', 'bot-iot_final.csv'])\nelse:\n    print('Skipped comparision.py. Set RUN_COMPARISON = True when bot-iot and bot-iot_final.csv are available.')

In [ ]:
# Step 3: train and evaluate the XGBoost pipeline.\nRUN_XGBOOST = True\n\nif RUN_XGBOOST:\n    run_step('testing_data_xgboost', f'{sys.executable} testing_data_Xgboost.py', required_paths=['processed_datasets/ton-iot_final.csv'])\nelse:\n    print('Skipped testing_data_Xgboost.py.')

In [ ]:
import json\nfrom pathlib import Path\n\nmanifest = {\n    'repo': REPO_URL,\n    'results': sorted(str(p) for p in RESULTS_DIR.glob('*')),\n    'generated_files': sorted(str(p) for p in Path('.').glob('*.pkl')) + sorted(str(p) for p in Path('plots').glob('*.png')),\n}\n\nPath('results/manifest.json').write_text(json.dumps(manifest, indent=2))\nprint(json.dumps(manifest, indent=2))

## After Running\n\nSend Codex the contents of `results/manifest.json` and any relevant `results/*.log` output, or commit small result files back to GitHub. Codex can then analyze the run and modify the project files for the next iteration.